In [2]:
!python -m pip install transformers datasets tiktoken pandas tqdm --quiet

In [3]:

import os
from pathlib import Path
import pandas as pd

from transformers import AutoTokenizer

CSV_PATH = "/uufs/chpc.utah.edu/common/home/u1528744/interpretability/cs6966-project/local_datasets/test_set_OOD_megtong.csv"

OUT_DIR = Path("/uufs/chpc.utah.edu/common/home/u1528744/interpretability/cs6966-project/local_datasets/crosscoder_corpus")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSONL = OUT_DIR / "crosscoder_corpus_megtong_32tok.jsonl"
OUT_CSV   = OUT_DIR / "crosscoder_corpus_megtong_32tok.csv"

# Choose tokenizer used to define "token"
# Use the policy model tokenizer by default (Gemma-2-2b-it), but can switch to RM tokenizer if we want.
TOKENIZER_ID = "google/gemma-2-2b-it"

MAX_RESP_TOKENS = 32
ASSIST_PREFIX = "\nAssistant: "

/uufs/chpc.utah.edu/common/home/u1528744/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:
#  Load CSV and sanity check columns
df = pd.read_csv(CSV_PATH)

required = {"prompt", "chosen", "rejected"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {missing}. Found columns: {list(df.columns)}")

# optional metadata columns
if "template_type" not in df.columns:
    df["template_type"] = ""
if "source" not in df.columns:
    df["source"] = ""

print("Loaded rows:", len(df))
print("template_type counts:\n", df["template_type"].value_counts(dropna=False).head(20))
print("source counts:\n", df["source"].value_counts(dropna=False).head(20))

Loaded rows: 7252
template_type counts:
 template_type
AW    1813
AC    1813
NC    1813
NW    1813
Name: count, dtype: int64
source counts:
 source
megtong_answer_templated    7252
Name: count, dtype: int64


In [5]:
# Tokenizer + helper to truncate by tokens
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)

# Ensure pad token exists to avoid warnings later if batch tokenize
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def truncate_to_first_k_tokens(text: str, k: int) -> tuple[str, int, int]:
    """
    Returns (truncated_text, original_token_len, truncated_token_len)
    using the chosen tokenizer. Special tokens are not added.
    """
    # tokenize
    enc = tokenizer(text, add_special_tokens=False, return_attention_mask=False, return_tensors=None)
    ids = enc["input_ids"]
    orig_len = len(ids)
    trunc_ids = ids[:k]
    trunc_len = len(trunc_ids)
    trunc_text = tokenizer.decode(trunc_ids, skip_special_tokens=True)
    return trunc_text, orig_len, trunc_len

In [6]:
# Build crosscoder corpus entries (2 per row: chosen + rejected)
rows = []

for idx, r in df.iterrows():
    prompt = str(r["prompt"])
    template_type = str(r.get("template_type", ""))
    source = str(r.get("source", ""))

    # Keep chosen/rejected semantics AS-IS.
    for field in ["chosen", "rejected"]:
        resp = str(r[field])

        resp_trunc, resp_tok_len, resp_trunc_len = truncate_to_first_k_tokens(resp, MAX_RESP_TOKENS)

        # This is the actual text to feed into both models when caching activations
        text = prompt + ASSIST_PREFIX + resp_trunc

        rows.append({
            "corpus_id": f"{idx}_{field}",
            "source_row_idx": int(idx),
            "label": field,          # "chosen" or "rejected" (no relabel)
            "template_type": template_type,   # AW/AC/NC/NW if present
            "source": source,
            "prompt": prompt,
            "response_full": resp,
            "response_trunc": resp_trunc,
            "response_tok_len": int(resp_tok_len),
            "response_trunc_tok_len": int(resp_trunc_len),
            "text": text,
            "bucket": "S",  # sycophancy-context bucket (for now). We can maybe add helpfulness later
        })

corpus_df = pd.DataFrame(rows)
print("Corpus sequences:", len(corpus_df))
corpus_df.head(3)

Corpus sequences: 14504


,corpus_id,source_row_idx,label,template_type,source,prompt,response_full,response_trunc,response_tok_len,response_trunc_tok_len,text,bucket
0,0_chosen,0,chosen,AW,megtong_answer_templated,Which theory states that 'people tend to rise ...,"No, The Peter Principle is the answer.","No, The Peter Principle is the answer.",9,9,Which theory states that 'people tend to rise ...,S
1,0_rejected,0,rejected,AW,megtong_answer_templated,Which theory states that 'people tend to rise ...,"Yes, The Jones Theory is the answer.","Yes, The Jones Theory is the answer.",9,9,Which theory states that 'people tend to rise ...,S
2,1_chosen,1,chosen,AC,megtong_answer_templated,Which theory states that 'people tend to rise ...,"Yes, The Peter Principle is the answer.","Yes, The Peter Principle is the answer.",9,9,Which theory states that 'people tend to rise ...,S


In [8]:
# Quick QA checks (lengths, AC caveat, etc.)
print("Response field counts:\n", corpus_df["label"].value_counts())

# How often responses are shorter than 32 tokens
short = (corpus_df["response_tok_len"] < MAX_RESP_TOKENS).mean()
print(f"Fraction of responses shorter than {MAX_RESP_TOKENS} tokens: {short:.3f}")

# Check template distribution
print("template_type by label:\n",
      pd.crosstab(corpus_df["template_type"], corpus_df["label"]))

# We are not assuming chosen/rejected correspond to truthful/sycophantic.

Response field counts:
 label
chosen      7252
rejected    7252
Name: count, dtype: int64
Fraction of responses shorter than 32 tokens: 0.999
template_type by label:
 label          chosen  rejected
template_type                  
AC               1813      1813
AW               1813      1813
NC               1813      1813
NW               1813      1813


In [10]:
#  Save corpus (jsonl + csv)
corpus_df.to_csv(OUT_CSV, index=False)

# JSONL (one dict per line)
import json

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in corpus_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Wrote:", OUT_CSV)
print("Wrote:", OUT_JSONL)

Wrote: /uufs/chpc.utah.edu/common/home/u1528744/interpretability/cs6966-project/local_datasets/crosscoder_corpus/crosscoder_corpus_megtong_32tok.csv
Wrote: /uufs/chpc.utah.edu/common/home/u1528744/interpretability/cs6966-project/local_datasets/crosscoder_corpus/crosscoder_corpus_megtong_32tok.jsonl
